### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="ieee_fraud_detection",
    dataset_year="2019",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/ieee-fraud-detection/overview",
    download_description=r"""
kaggle competitions download -c ieee-fraud-detection -p local-data-warehouse/ieee_fraud_detection \
&& unzip local-data-warehouse/ieee_fraud_detection/ieee-fraud-detection.zip -d local-data-warehouse/ieee_fraud_detection/ \
&& rm local-data-warehouse/ieee_fraud_detection/ieee-fraud-detection.zip
""",
    # References
    academic_reference_bibtex=r"""@misc{ieee-fraud-detection,
    author = {Addison Howard and Bernadette Bouchon-Meunier and IEEE CIS and inversion and John Lei and Lynn@Vesta and Marcus2010 and Prof. Hussein Abbass},
    title = {IEEE-CIS Fraud Detection},
    year = {2019},
    howpublished = {\url{https://kaggle.com/competitions/ieee-fraud-detection}},
    note = {Kaggle}
}
""",
    academic_reference_bibtex_key="ieee-fraud-detection",
    license="non-commercial purposes only, including academic research",
    data_tags=["Non-IID", "Temporal"],
    curation_comments="""
- We use insights of the first place solution on Kaggle for conceptualizing the task: https://www.kaggle.com/competitions/ieee-fraud-detection/discussion/111284
- The data is given as transactions, but the task is to predict fraudulent clients. Once a transaction is detected as fraud, the entire account is considered fraudulent.
- The competition host commented on the labeling logic: "The logic of our labeling is define reported chargeback on the card as fraud transaction (isFraud=1) and transactions posterior to it with either user account, email address or billing address directly linked to these attributes as fraud too" (https://www.kaggle.com/c/ieee-fraud-detection/discussion/101203#589276)
- We load train_transaction.csv and train_identity.csv and merge them on TransactionID.
- We derive Transaction_date from TransactionDT using the competition reference start date 2017-11-30.
- The Kaggle competition also provides test_transaction.csv and test_identity.csv, but they are unlabeled and are therefore excluded from df for curation checks.
- We normalize the D-columns, since they correspond to days since a certain event, we can normalize them by the transaction date to get the actual day of the event. This is also what the first place solution on Kaggle did.
- We add a uid feature used by the first place solution on Kaggle to identify unique users. Note that this feature is not perfect and just an approximation by the Kaggle users. Moreover, due to missing values, for ~15% of the samples no uid could be reconstructed.
- Following the first place Kaggle solution, which used month-based cross-validation, we create splits using forecasting horizons of 1 month.
- The competition test data left a planning window of one month, so do we.
- Anomaly: Missing values.
- Anomaly: Several features might allow to approximate to the uid and can lead to overfitting.
- Anomaly: Grouped data with members of a group often sharing the same label, depending on whether fraud occurred previously or not.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="isFraud",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    # For classification
    stratify_on="isFraud",
    # For time
    time_on="Transaction_date",
)

## Preprocessing

In [2]:
import pandas as pd
import datetime
import numpy as np

# LOAD TRAIN
df = pd.read_csv(dataset_mold.path / 'train_transaction.csv', index_col='TransactionID', engine="pyarrow")
train_id = pd.read_csv(dataset_mold.path / 'train_identity.csv', index_col='TransactionID', engine="pyarrow")
df = df.merge(train_id, how='left', left_index=True, right_index=True)

# LOAD TEST
# X_test = pd.read_csv(dataset_mold.path / 'test_transaction.csv', index_col='TransactionID', engine="pyarrow")
# test_id = pd.read_csv(dataset_mold.path / 'test_identity.csv', index_col='TransactionID', engine="pyarrow")
# fix = {o:n for o, n in zip(test_id.columns, train_id.columns)}
# test_id.rename(columns=fix, inplace=True)
# df_test = X_test.merge(test_id, how='left', left_index=True, right_index=True)

START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')
df['Transaction_date'] = df['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds=x)))
# df['DT_M'] = (df['Transaction_date'].dt.year - 2017) * 12 + df['Transaction_date'].dt.month

df['day'] = df['TransactionDT'] / (24 * 60 * 60)

# NORMALIZE D COLUMNS, since they correspond to days since a certain event, we can normalize them by the transaction date to get the actual day of the event. This is also what the first place solution on Kaggle did.
for i in range(1,16):
    if i in [1,2,3,5,9]: continue
    df['D'+str(i)] =  df['D'+str(i)] - df['TransactionDT']/np.float32(24*60*60)

df = df.drop(columns=['TransactionDT'])

# This is the uid that as used for feature engineering in the first place solution on Kaggle. They also have a better way to find uids 
# df['uid'] = df['card1'].astype(str) + '_' + df['addr1'].astype(str)  + '_' + np.floor(df.day - df.D1).astype(str)
# df.loc[np.logical_or(df['D1'].isna(), df['addr1'].isna()),["uid"]] = np.nan
# Use the uid version that has the best missing values (15%) vs. accuracy for same uid (99.96%)
uids = pd.read_csv('uids_v1_no_multiuid_cleaning.csv',usecols=['TransactionID','uid'],engine="pyarrow")
df = df.merge(uids,on='TransactionID',how='left')

# Obtain correct cat indices
cat_cols = ['ProductCD',"addr1","addr2","P_emaildomain","R_emaildomain","DeviceType","DeviceInfo"]
cat_cols += [f"card{i}" for i in range(1,7)]
cat_cols += [f"M{i}" for i in range(1,10)]
cat_cols += [f"id_{i}" for i in range(12,39)]
cat_cols += ["uid"]

for col in cat_cols:
    df[col] = df[col].astype('category')

df["isFraud"] = df["isFraud"].astype("category")
df = df.sort_values(by=["Transaction_date"]).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 590,540
Columns: 436
Use sampling: False (sample size: 590,540)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['TransactionID', 'day', 'Transaction_date', 'D10', 'D15', 'D4', 'D11', 'uid', 'id_02', 'D8']
Rows remaining as candidates after top-10 filter: 0 (of 590,540)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,TransactionID,isFraud,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,M1,M2,M3,M4,M5,M6,M7,M8,M9,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30,V31,V32,V33,V34,V35,V36,V37,V38,V39,V40,V41,V42,V43,V44,V45,V46,V47,V48,V49,V50,V51,V52,V53,V54,V55,V56,V57,V58,V59,V60,V61,V62,V63,V64,V65,V66,V67,V68,V69,V70,V71,V72,V73,V74,V75,V76,V77,V78,V79,V80,V81,V82,V83,V84,V85,V86,V87,V88,V89,V90,V91,V92,V93,V94,V95,V96,V97,V98,V99,V100,V101,V102,V103,V104,V105,V106,V107,V108,V109,V110,V111,V112,V113,V114,V115,V116,V117,V118,V119,V120,V121,V122,V123,V124,V125,V126,V127,V128,V129,V130,V131,V132,V133,V134,V135,V136,V137,V138,V139,V140,V141,V142,V143,V144,V145,V146,V147,V148,V149,V150,V151,V152,V153,V154,V155,V156,V157,V158,V159,V160,V161,V162,V163,V164,V165,V166,V167,V168,V169,V170,V171,V172,V173,V174,V175,V176,V177,V178,V179,V180,V181,V182,V183,V184,V185,V186,V187,V188,V189,V190,V191,V192,V193,V194,V195,V196,V197,V198,V199,V200,V201,V202,V203,V204,V205,V206,V207,V208,V209,V210,V211,V212,V213,V214,V215,V216,V217,V218,V219,V220,V221,V222,V223,V224,V225,V226,V227,V228,V229,V230,V231,V232,V233,V234,V235,V236,V237,V238,V239,V240,V241,V242,V243,V244,V245,V246,V247,V248,V249,V250,V251,V252,V253,V254,V255,V256,V257,V258,V259,V260,V261,V262,V263,V264,V265,V266,V267,V268,V269,V270,V271,V272,V273,V274,V275,V276,V277,V278,V279,V280,V281,V282,V283,V284,V285,V286,V287,V288,V289,V290,V291,V292,V293,V294,V295,V296,V297,V298,V299,V300,V301,V302,V303,V304,V305,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,id_20,id_21,id_22,id_23,id_24,id_25,id_26,id_27,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo,Transaction_date,day,uid
0,2987000,0,68.5,W,13926,NaN,150.0,discover,142.0,credit,315.0,87.0,19.0,NaN,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,1.0,1.0,14.0,NaN,13.0,NaN,NaN,NaN,NaN,NaN,NaN,12.000000,12.000000,NaN,NaN,NaN,-1.000000,T,T,T,M2,F,T,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-12-01 00:00:00,1.000000,2987000.0
1,2987001,0,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0,87

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,id_24,category,585793.0,99.20,12.0,"11.0, 15.0, 16.0, 18.0, 21.0, 24.0, 17.0, 26.0, 25.0, 19.0"
1,id_21,category,585381.0,99.13,490.0,"252.0, 228.0, 255.0, 596.0, 576.0, 849.0, 277.0, 755.0, 848.0, 668.0"
2,id_25,category,585408.0,99.13,341.0,"321.0, 205.0, 426.0, 501.0, 371.0, 524.0, 442.0, 509.0, 123.0, 126.0"
3,id_26,category,585377.0,99.13,95.0,"161.0, 184.0, 142.0, 102.0, 100.0, 119.0, 169.0, 147.0, 215.0, 121.0"
4,id_22,category,585371.0,99.12,25.0,"14.0, 41.0, 33.0, 21.0, 17.0, 39.0, 36.0, 22.0, 12.0, 31.0"
5,id_23,category,585371.0,99.12,3.0,"IP_PROXY:TRANSPARENT, IP_PROXY:ANONYMOUS, IP_PROXY:HIDDEN"
6,id_27,category,585371.0,99.12,2.0,"Found, NotFound"
7,id_18,category,545427.0,92.36,18.0,"15.0, 13.0, 12.0, 18.0, 20.0, 17.0, 26.0, 21.0, 24.0, 11.0"
8,id_33,category,517251.0,87.59,260.0,"1920x1080, 1366x768, 1334x750, 2208x1242, 1440x900, 1600x900, 2048x1536, 1280x800, 2560x1600, 2560x1440"
9,id_30,category,512975.0,86.87,75.0,"Windows 10, Windows 7, iOS 11.2.1, iOS 11.1.2, Android 7.0, Mac OS X 10_12_6, Mac OS X 10_11_6, iOS 11.3.0, Windows 8.1, Mac OS X 10_10_5"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
TransactionID,590540.0,3.282270e+06,170474.358321,2.987000e+06,3.577539e+06
TransactionAmt,590540.0,1.350272e+02,239.162522,2.510000e-01,3.193739e+04
dist1,238269.0,1.185022e+02,371.872026,0.000000e+00,1.028600e+04
dist2,37627.0,2.318554e+02,529.053494,0.000000e+00,1.162300e+04
C1,590540.0,1.409246e+01,133.569018,0.000000e+00,4.685000e+03
C2,590540.0,1.526973e+01,154.668899,0.000000e+00,5.691000e+03
C3,590540.0,5.643987e-03,0.150536,0.000000e+00,2.600000e+01
C4,590540.0,4.092185e+00,68.848459,0.000000e+00,2.253000e+03
C5,590540.0,5.571526e+00,25.786976,0.000000e+00,3.490000e+02
C6,590540.0,9.071082e+00,71.508467,0.000000e+00,2.253000e+03


In [7]:
# Categorical Feature Statistics
cat_stats

value   count    pct
column           rank                                      
DeviceInfo       1                      <NA>  471874  79.91
                 2                   Windows   47722   8.08
                 3                iOS Device   19782   3.35
                 4                     MacOS   12573   2.13
                 5               Trident/7.0    7440   1.26
DeviceType       1                      <NA>  449730  76.16
                 2                   desktop   85165  14.42
                 3                    mobile   55645   9.42
M1               1                         T  319415  54.09
                 2                      <NA>  271100  45.91
                 3                         F      25   0.00
M2               1                         T  285468  48.34
                 2                      <NA>  271100  45.91
                 3                         F   33972   5.75
M3               1                      <NA>  271100  45.91
                 2                         T  251731  42.63
                 3                         F   67709  11.47
M4               1                      <NA>  281444  47.66
                 2                        M0  196405  33.26
                 3                        M2   59865  10.14
                 4                        M1   52826   8.95
M5               1                      <NA>  350482  59.35
                 2                         F  132491  22.44
                 3                         T  107567  18.22
M6               1                         F  227856  38.58
                 2                         T  193324  32.74
                 3                      <NA>  169360  28.68
M7               1                      <NA>  346265  58.64
                 2                         F  211374  35.79
                 3                         T   32901   5.57
M8               1                      <NA>  346252  58.63
                 2                         F  155251  26.29
                 3                         T   89037  15.08
M9               1                      <NA>  346252  58.63
                 2                         T  205656  34.83
                 3                         F   38632   6.54
P_emaildomain    1                 gmail.com  228355  38.67
                 2                 yahoo.com  100934  17.09
                 3                      <NA>   94456  15.99
                 4               hotmail.com   45250   7.66
                 5             anonymous.com   36998   6.27
ProductCD        1                         W  439670  74.45
                 2                         C   68519  11.60
                 3                         R   37699   6.38
                 4                         H   33024   5.59
                 5                         S   11628   1.97
R_emaildomain    1                      <NA>  453249  76.75
                 2                 gmail.com   57147   9.68
                 3               hotmail.com   27509   4.66
                 4             anonymous.com   20529   3.48
                 5                 yahoo.com   11842   2.01
Transaction_date 1       2018-03-19 15:53:37       8   0.00
                 2       2018-01-19 21:24:26       5   0.00
                 3       2018-04-12 23:49:11       5   0.00
                 4       2018-02-21 18:09:48       5   0.00
                 5       2017-12-23 23:54:56       4   0.00
addr1            1                      <NA>   65706  11.13
                 2                     299.0   46335   7.85
                 3                     325.0   42751   7.24
                 4                     204.0   42020   7.12
                 5                     264.0   39870   6.75
addr2            1                      87.0  520481  88.14
                 2                      <NA>   65706  11.13
                 3                      60.0    3084   0.52
                 4                      96.0     638   0.11
                 5 

In [8]:
# Target Distribution
target_df

,count,pct
isFraud,,
0,569877,96.5
1,20663,3.5


## Task Curation

In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata

date_col = task_mold.time_on
target_col = task_mold.target_column_name

months = (df['Transaction_date'].dt.year - 2017) * 12 + df['Transaction_date'].dt.month

# -- For Temporal non-IID data -> manual processing required
splits = {
    0: {
        0: ((months[months<=15]).index.to_list(), (months[months==17]).index.to_list()),
    },
    1: {
        0: ((months[months<=14]).index.to_list(), (months[months==16]).index.to_list()),
    },
    2: {
        0: ((months[months<=13]).index.to_list(), (months[months==15]).index.to_list()),
    },
}

used_in_train = set()
used_in_test = set()
used_data = set()
for split in splits:
    train_idx, test_idx = splits[split][0]
    used_in_train.update(train_idx)
    used_in_test.update(test_idx)
    used_data.update(train_idx)
    used_data.update(test_idx)

    print(f"\n=== Step {split} ===")
    print("Train size:", len(train_idx), "| Test size:", len(test_idx))
    print("Train target mean:", df.loc[train_idx, target_col].astype(int).mean())
    print("Test target mean:", df.loc[test_idx, target_col].astype(int).mean())
    print("% test samples with reoccuring, non-missing uids:", df["uid"].dropna().isin(set(df.loc[test_idx, "uid"]).intersection(set(df.loc[train_idx, "uid"]))).mean())
    print("% test samples with missing uids:", df["uid"].isna().mean())

    assert len(set(train_idx).intersection(set(test_idx)))==0, "Train and test indices overlap!"

print(f"{len(used_data)/df.shape[0]:.4f} of the samples are used.")
print(f"{len(used_in_train)/df.shape[0]:.4f} of the samples are used in training")
print(f"{len(used_in_test)/df.shape[0]:.4f} of the samples are used in testing.")

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="To define three train/test splits, we use the last three months as test sets, leave 1 month as the planning gap, and use the remaining training data.",
    splits=splits,
    # For time data
    time_horizon="1",
    time_horizon_unit="months",
)


=== Step 0 ===
Train size: 417559 | Test size: 89326
Train target mean: 0.03525489811020718
Test target mean: 0.03486107068490697
% test samples with reoccuring, non-missing uids: 0.11205089700142942
% test samples with missing uids: 0.15652792359535342

=== Step 1 ===
Train size: 315927 | Test size: 83655
Train target mean: 0.03387491414155802
Test target mean: 0.03380551072858765
% test samples with reoccuring, non-missing uids: 0.11998699066861539
% test samples with missing uids: 0.15652792359535342

=== Step 2 ===
Train size: 229906 | Test size: 101632
Train target mean: 0.03155637521421799
Test target mean: 0.03954463161209068
% test samples with reoccuring, non-missing uids: 0.12339591731847165
% test samples with missing uids: 0.15652792359535342
1.0000 of the samples are used.
0.7071 of the samples are used in training
0.4650 of the samples are used in testing.


## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to ieee_fraud_detection/019db516-2f8e-7e50-a8c4-f1c57754f52c
019db516-2f8e-7e50-a8c4-f1c57754f52c
2368c2e67810ef4127ceebe3023d115366be24d5f9ceeb87319774a1a4e6c93d
